# 09 — Bayesian Regression
**References:** Gelman et al. BDA3 Ch. 14 · Gelman & Hill (2007) Ch. 3 · Carvalho, Polson & Scott (2010)

## Narrative thread
```
Bayesian OLS -> Conjugate Normal-IG prior -> Ridge as Gaussian prior -> Lasso as Laplace -> Horseshoe prior -> Variable selection
```

## 1. Bayesian linear regression

$$\mathbf{y} = \mathbf{X}\boldsymbol{\beta} + \boldsymbol{\varepsilon}, \quad \boldsymbol{\varepsilon} \sim \mathcal{N}(\mathbf{0}, \sigma^2 \mathbf{I})$$

**Conjugate prior** (Normal-Inverse-Gamma):
$$\boldsymbol{\beta} \mid \sigma^2 \sim \mathcal{N}(\mathbf{b}_0, \sigma^2 \mathbf{B}_0), \qquad \sigma^2 \sim \text{IG}(a_0, c_0)$$

**Posterior:**
$$\boldsymbol{\beta} \mid \mathbf{y}, \sigma^2 \sim \mathcal{N}(\mathbf{b}_n, \sigma^2 \mathbf{B}_n)$$
$$\mathbf{B}_n = (\mathbf{B}_0^{-1} + \mathbf{X}^\top\mathbf{X})^{-1}, \qquad \mathbf{b}_n = \mathbf{B}_n(\mathbf{B}_0^{-1}\mathbf{b}_0 + \mathbf{X}^\top\mathbf{y})$$

**Key insight:** The posterior mean $\mathbf{b}_n$ is a precision-weighted combination of prior information and OLS. With a flat prior ($\mathbf{B}_0^{-1} \to \mathbf{0}$), $\mathbf{b}_n \to \hat{\boldsymbol{\beta}}_{\text{OLS}}$.

**Regularization as prior:**
- Ridge: $\boldsymbol{\beta} \sim \mathcal{N}(\mathbf{0}, \lambda^{-1}\mathbf{I})$ — Gaussian prior = $\ell_2$ penalty
- Lasso: $\boldsymbol{\beta} \sim \text{Laplace}(0, \lambda^{-1})$ — double exponential = $\ell_1$ penalty
- Elastic net: mixture of Gaussian and Laplace priors

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

# ── Bayesian regression with credible intervals ────────────────────────────
# Simple regression: y = 2 + 0.8*x + eps
n = 40
x = np.random.uniform(-3, 3, n)
y = 2 + 0.8*x + np.random.normal(0, 1, n)
X = np.column_stack([np.ones(n), x])

# Prior: beta ~ N(0, 10^2 * I),  sigma^2 ~ IG(1, 1)
B0_inv = np.eye(2) / 100     # weak prior precision
b0     = np.zeros(2)
a0, c0 = 1.0, 1.0

# Posterior
Bn_inv = B0_inv + X.T @ X
Bn     = np.linalg.inv(Bn_inv)
bn     = Bn @ (B0_inv @ b0 + X.T @ y)
an     = a0 + n/2
cn     = c0 + 0.5*(y @ y + b0 @ B0_inv @ b0 - bn @ Bn_inv @ bn)

# Posterior mean of sigma^2
sigma2_post_mean = cn / (an - 1)

# Marginal posterior of beta: multivariate t
# Posterior predictive for new x values
x_pred = np.linspace(-4, 4, 200)
X_pred = np.column_stack([np.ones(200), x_pred])

# Posterior mean line
y_pred_mean = X_pred @ bn

# Credible intervals via Monte Carlo
S = 5000
# Draw sigma^2 from IG(an, cn)
sigma2_draws = 1/np.random.gamma(an, 1/cn, S)
# Draw beta | sigma^2
beta_draws = np.array([np.random.multivariate_normal(bn, s2*Bn) for s2 in sigma2_draws])
# Posterior predictive
y_draws = beta_draws @ X_pred.T + np.random.normal(0, np.sqrt(sigma2_draws[:, None]), (S, 200))

ci_lo = np.percentile(y_draws, 2.5, axis=0)
ci_hi = np.percentile(y_draws, 97.5, axis=0)
mu_lo = np.percentile(beta_draws @ X_pred.T, 2.5, axis=0)
mu_hi = np.percentile(beta_draws @ X_pred.T, 97.5, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax0 = axes[0]
ax0.scatter(x, y, color='#f72585', s=40, alpha=0.7, zorder=5, label='Data')
ax0.plot(x_pred, y_pred_mean, color='#4361ee', lw=2.5, label='Posterior mean')
ax0.fill_between(x_pred, mu_lo, mu_hi, alpha=0.3, color='#4361ee', label='95% CI for $E[y|x]$')
ax0.fill_between(x_pred, ci_lo, ci_hi, alpha=0.15, color='#4361ee', label='95% Posterior predictive')
ax0.plot(x_pred, 2 + 0.8*x_pred, 'k--', lw=1.5, label='True line')
ax0.set_xlabel('x'); ax0.set_ylabel('y')
ax0.set_title(f'Bayesian linear regression\nPost. mean: $\\hat\\alpha={bn[0]:.2f}$, $\\hat\\beta={bn[1]:.2f}$')
ax0.legend(fontsize=7)

# Joint posterior of (alpha, beta)
ax1 = axes[1]
ax1.scatter(beta_draws[:500, 0], beta_draws[:500, 1], s=4, alpha=0.3, color='#4361ee')
ax1.scatter(*bn, color='#f72585', s=100, zorder=5, label=f'Post. mean ({bn[0]:.2f},{bn[1]:.2f})')
ax1.scatter(2, 0.8, color='#06d6a0', s=100, marker='*', zorder=5, label='True (2, 0.8)')
ax1.set_xlabel('$\\alpha$ (intercept)'); ax1.set_ylabel('$\\beta$ (slope)')
ax1.set_title('Joint posterior of $(\\alpha, \\beta)$\nCorrelation reflects confounding in design')
ax1.legend(fontsize=8)

plt.suptitle('Bayesian Linear Regression: Posterior and Predictive Intervals', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. The Horseshoe prior for sparse regression

In high-dimensional regression ($p \gg n$), most coefficients are zero. The **horseshoe prior** (Carvalho et al. 2010) achieves strong shrinkage for small coefficients while allowing large signals to pass through:

$$\beta_j \mid \lambda_j, \tau \sim \mathcal{N}(0, \lambda_j^2 \tau^2)$$
$$\lambda_j \sim \text{Half-Cauchy}(0, 1) \qquad \text{(local shrinkage)}$$
$$\tau \sim \text{Half-Cauchy}(0, 1) \qquad \text{(global shrinkage)}$$

**Shrinkage profile:** The posterior mean under horseshoe is:
$$E[\beta_j \mid y] \approx (1 - \kappa_j)\,\hat\beta_j^{\text{OLS}}, \qquad \kappa_j = \frac{1}{1 + n\lambda_j^2\tau^2}$$

Signals with $\hat\beta_j \gg \tau$ get $\kappa_j \approx 0$ (no shrinkage); noise with $\hat\beta_j \approx 0$ get $\kappa_j \approx 1$ (full shrinkage). The horseshoe achieves near-minimax optimal estimation for sparse signals.

In [ ]:
# ── Shrinkage profiles: Ridge vs Lasso vs Horseshoe ───────────────────────
beta_ols = np.linspace(-6, 6, 300)  # hypothetical OLS estimates
tau       = 1.0  # global scale

# Ridge shrinkage: beta_post = (1/(1+lambda)) * beta_ols
ridge_shrink = lambda b, lam: b / (1 + lam)

# Lasso (soft thresholding)
lasso_shrink = lambda b, lam: np.sign(b) * np.maximum(np.abs(b) - lam, 0)

# Horseshoe: approximate via expected shrinkage factor
# E[1 - kappa_j | beta_ols] ≈ min(1, beta_ols^2 / tau^2)
horseshoe_shrink = lambda b, tau: np.where(
    np.abs(b) < tau,
    b * (b/tau)**2 / (1 + (b/tau)**2),
    b * (1 - 1/(1 + (b/tau)**4))
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax0 = axes[0]
ax0.plot(beta_ols, beta_ols,                           color='gray',    lw=1.5, linestyle=':', label='No shrinkage (OLS)')
ax0.plot(beta_ols, ridge_shrink(beta_ols, 1.0),        color='#4361ee', lw=2.5, label='Ridge (λ=1)')
ax0.plot(beta_ols, lasso_shrink(beta_ols, 1.0),        color='#f72585', lw=2.5, label='Lasso (λ=1)')
ax0.plot(beta_ols, horseshoe_shrink(beta_ols, tau),    color='#06d6a0', lw=2.5, label=f'Horseshoe (τ={tau})')
ax0.set_xlabel('OLS estimate $\\hat\\beta$'); ax0.set_ylabel('Posterior mean')
ax0.set_title('Shrinkage profiles: Ridge vs Lasso vs Horseshoe\n'
               'Horseshoe: strong shrink near 0, near-no-shrink for large signals')
ax0.legend(); ax0.set_xlim(-6, 6); ax0.set_ylim(-6, 6)

# Prior density comparison
b_g = np.linspace(-5, 5, 400)
ax1 = axes[1]
ax1.plot(b_g, stats.norm.pdf(b_g, 0, 1),            color='#4361ee', lw=2.5, label='Ridge (Normal prior)')
ax1.plot(b_g, stats.laplace.pdf(b_g, 0, 1),          color='#f72585', lw=2.5, label='Lasso (Laplace prior)')
# Horseshoe marginal (approximate via mixture)
lam_draws = np.abs(np.random.standard_cauchy(50_000))
beta_hs   = np.random.normal(0, lam_draws * tau)
ax1.hist(beta_hs, bins=200, density=True, range=(-5,5), color='#06d6a0',
         alpha=0.6, edgecolor='none', label='Horseshoe (marginal)')
ax1.set_xlabel('$\\beta$'); ax1.set_ylabel('Density'); ax1.set_yscale('log')
ax1.set_title('Prior densities (log scale)\nHorseshoe: heavy tails + spike at 0')
ax1.set_ylim(1e-4, 2); ax1.legend()

plt.suptitle('Bayesian Regularization: Prior Shapes and Shrinkage Profiles', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Key takeaways

| Prior on $\boldsymbol{\beta}$ | Frequentist equivalent | Induces | Best for |
|---|---|---|---|
| $\mathcal{N}(0, \lambda^{-1}I)$ | Ridge / $\ell_2$ | Global shrinkage | Moderate coefficients |
| Laplace$(0, \lambda^{-1})$ | Lasso / $\ell_1$ | Sparsity via thresholding | Many zeros |
| Horseshoe | — | Spike + slab: adaptive | Truly sparse signal |
| Spike-and-slab | Subset selection | Hard sparsity | Variable selection |

**Next:** notebook 10 — variational inference: fast approximate Bayesian inference.